# Synthetic Viral Amplicon Sequencing for Outbreak Lineage Assignment

I am treating this notebook as a small surveillance analysis for a synthetic viral amplicon sequencing run. The project `SYN-VLA-2026-Q2` contains 12 synthetic specimens collected across three sites, and the goal is to assign outbreak lineages from mutation presence/absence calls while flagging specimens whose amplicon coverage is too low for confident interpretation.

All input data below are generated in the notebook with a fixed random seed. The samples, sites, read depth metrics, mutations, and lineage calls are synthetic and are not for clinical use.

In [2]:
import random
from collections import Counter

rng = random.Random(20240602)

PROJECT_ID = "SYN-VLA-2026-Q2"
MUTATION_MARKERS = ["S:D614G", "ORF1a:P323L", "N:R203K", "S:A701V"]
AMPLICON_DEPTH_COLUMNS = [
    "amplicon_S_614",
    "amplicon_ORF1a_323",
    "amplicon_N_203",
    "amplicon_S_701",
]

metadata_records = [
    ("VLA-S001", "NorthClinic", "2026-W14", "VX.1"),
    ("VLA-S002", "HarborLab", "2026-W14", "VX.2"),
    ("VLA-S003", "NorthClinic", "2026-W14", "VX.2.1"),
    ("VLA-S004", "RidgeMobile", "2026-W14", "VX.2"),
    ("VLA-S005", "HarborLab", "2026-W15", "VX.2.1"),
    ("VLA-S006", "RidgeMobile", "2026-W15", "VX.2.1"),
    ("VLA-S007", "NorthClinic", "2026-W15", "VX.1"),
    ("VLA-S008", "HarborLab", "2026-W15", "VX.2.1"),
    ("VLA-S009", "RidgeMobile", "2026-W16", "VX.2.1"),
    ("VLA-S010", "NorthClinic", "2026-W16", "VX.2"),
    ("VLA-S011", "HarborLab", "2026-W16", "VX.2.1"),
    ("VLA-S012", "RidgeMobile", "2026-W16", "VX.1"),
]

base_depths = [
    [430, 452, 421, 448], [510, 533, 506, 521], [620, 601, 635, 612],
    [58, 72, 64, 69], [575, 590, 568, 584], [640, 626, 651, 633],
    [405, 399, 418, 411], [700, 684, 692, 711], [77, 88, 71, 82],
    [485, 470, 492, 479], [615, 604, 629, 610], [52, 65, 58, 61],
]

lineage_marker_sets = {
    "VX.1": {"S:D614G", "ORF1a:P323L"},
    "VX.2": {"S:D614G", "ORF1a:P323L", "N:R203K"},
    "VX.2.1": {"S:D614G", "ORF1a:P323L", "N:R203K", "S:A701V"},
}

samples = []
for (sample_id, site, week, designed_lineage), template in zip(metadata_records, base_depths):
    depths = [depth + rng.randint(-6, 6) for depth in template]
    present = lineage_marker_sets[designed_lineage]
    sample = {
        "project_id": PROJECT_ID,
        "sample_id": sample_id,
        "collection_site": site,
        "collection_week": week,
        "designed_lineage": designed_lineage,
        "mean_depth": round(sum(depths) / len(depths), 1),
        "min_depth": min(depths),
        "pct_amplicons_ge100": round(sum(depth >= 100 for depth in depths) / len(depths) * 100),
    }
    sample.update(dict(zip(AMPLICON_DEPTH_COLUMNS, depths)))
    for mutation in MUTATION_MARKERS:
        sample[mutation] = mutation in present
    samples.append(sample)

print(f"project_id {PROJECT_ID}")
print(f"synthetic_samples {len(samples)}")
print("mutation_markers " + ", ".join(MUTATION_MARKERS))
print("amplicons " + ", ".join(AMPLICON_DEPTH_COLUMNS))

project_id SYN-VLA-2026-Q2
synthetic_samples 12
mutation_markers S:D614G, ORF1a:P323L, N:R203K, S:A701V
amplicons amplicon_S_614, amplicon_ORF1a_323, amplicon_N_203, amplicon_S_701


## Surveillance goal and lineage rules

For this synthetic outbreak exercise I use a deliberately small mutation rule set so the lineage targets are easy to retrieve:

- `VX.1`: requires `S:D614G` and `ORF1a:P323L`; `N:R203K` is absent.
- `VX.2`: requires `S:D614G`, `ORF1a:P323L`, and `N:R203K`.
- `VX.2.1`: requires the `VX.2` pattern plus `S:A701V`, a synthetic sublineage marker.

I require a minimum per-sample amplicon depth of 100 reads and a mean depth of 150 reads before reporting the mutation-derived lineage.

In [3]:
def assign_lineage_from_mutations(
    present_mutations,
    min_depth,
    mean_depth,
    min_depth_threshold=100,
    mean_depth_threshold=150,
):
    """Assign a synthetic viral lineage from mutation calls and coverage metrics."""
    present = set(present_mutations)

    if min_depth < min_depth_threshold or mean_depth < mean_depth_threshold:
        return "QC_FAIL_LOW_COVERAGE"
    if {"S:D614G", "ORF1a:P323L", "N:R203K", "S:A701V"}.issubset(present):
        return "VX.2.1"
    if {"S:D614G", "ORF1a:P323L", "N:R203K"}.issubset(present):
        return "VX.2"
    if {"S:D614G", "ORF1a:P323L"}.issubset(present):
        return "VX.1"
    return "UNASSIGNED"

print("assign_lineage_from_mutations smoke_check")
print(assign_lineage_from_mutations(["S:D614G", "ORF1a:P323L"], 410, 430.0))
print(assign_lineage_from_mutations(["S:D614G", "ORF1a:P323L", "N:R203K"], 410, 430.0))
print(assign_lineage_from_mutations(["S:D614G", "ORF1a:P323L", "N:R203K", "S:A701V"], 410, 430.0))
print(assign_lineage_from_mutations(["S:D614G", "ORF1a:P323L", "N:R203K"], 59, 64.0))

assign_lineage_from_mutations smoke_check
VX.1
VX.2
VX.2.1
QC_FAIL_LOW_COVERAGE


In [4]:
def present_marker_list(sample):
    return [mutation for mutation in MUTATION_MARKERS if sample[mutation]]


def format_table(records, columns):
    widths = [max(len(str(row[column])) for row in records + [{column: column}]) for column in columns]

    def align(value, width):
        if isinstance(value, bool):
            return str(value).ljust(width)
        return str(value).rjust(width) if isinstance(value, (int, float)) else str(value).ljust(width)

    header = "  ".join(column.ljust(width) for column, width in zip(columns, widths))
    rule = "  ".join("-" * width for width in widths)
    rows = ["  ".join(align(row[column], width) for column, width in zip(columns, widths)) for row in records]
    return "\n".join([header, rule] + rows)


for sample in samples:
    sample["called_lineage"] = assign_lineage_from_mutations(
        present_marker_list(sample),
        sample["min_depth"],
        sample["mean_depth"],
    )
    sample["low_coverage"] = sample["called_lineage"] == "QC_FAIL_LOW_COVERAGE"

assignment_columns = [
    "sample_id", "collection_site", "mean_depth", "min_depth", "pct_amplicons_ge100",
    "S:D614G", "ORF1a:P323L", "N:R203K", "S:A701V", "called_lineage", "low_coverage",
]

assignment_table = []
for sample in samples:
    row = {column: sample[column] for column in assignment_columns}
    for mutation in MUTATION_MARKERS:
        row[mutation] = "Y" if sample[mutation] else "N"
    assignment_table.append(row)

print(format_table(assignment_table, assignment_columns))

sample_id  collection_site  mean_depth  min_depth  pct_amplicons_ge100  S:D614G  ORF1a:P323L  N:R203K  S:A701V  called_lineage        low_coverage
---------  ---------------  ----------  ---------  -------------------  -------  -----------  -------  -------  --------------------  ------------
VLA-S001   NorthClinic           440.5        427                  100  Y        Y            N        N        VX.1                  False       
VLA-S002   HarborLab             515.5        501                  100  Y        Y            Y        N        VX.2                  False       
VLA-S003   NorthClinic           617.2        601                  100  Y        Y            Y        Y        VX.2.1                False       
VLA-S004   RidgeMobile            64.0         59                    0  Y        Y            Y        N        QC_FAIL_LOW_COVERAGE  True        
VLA-S005   HarborLab             578.5        571                  100  Y        Y            Y        Y        VX.2.1

## Low-coverage sample handling

I do not force a lineage label onto low-coverage specimens. Instead, samples with any amplicon below 100 reads or mean depth below 150 reads are labeled `QC_FAIL_LOW_COVERAGE` and excluded from the site-level lineage prevalence summary. Their mutation calls remain visible in the assignment table so a scientist can decide whether to resequence, repeat the amplicon pool, or keep the sample as qualitative supporting evidence.

In [5]:
low_coverage_samples = [sample for sample in samples if sample["low_coverage"]]

print(f"low_coverage_samples {len(low_coverage_samples)}")
for sample in low_coverage_samples:
    print(
        f"{sample['sample_id']} {sample['collection_site']} "
        f"mean_depth={sample['mean_depth']} min_depth={sample['min_depth']}"
    )

low_coverage_samples 3
VLA-S004 RidgeMobile mean_depth=64.0 min_depth=59
VLA-S009 RidgeMobile mean_depth=80.8 min_depth=67
VLA-S012 RidgeMobile mean_depth=60.0 min_depth=49


In [6]:
lineage_order = ["VX.1", "VX.2", "VX.2.1"]
site_order = sorted({sample["collection_site"] for sample in samples})
lineage_counts_by_site = {site: {lineage: 0 for lineage in lineage_order} for site in site_order}

for sample in samples:
    if not sample["low_coverage"]:
        lineage_counts_by_site[sample["collection_site"]][sample["called_lineage"]] += 1

valid_lineage_counts = Counter(
    sample["called_lineage"] for sample in samples if not sample["low_coverage"]
)
dominant_lineage = max(lineage_order, key=lambda lineage: valid_lineage_counts[lineage])

print("collection_site   VX.1   VX.2 VX.2.1")
for site in site_order:
    counts = lineage_counts_by_site[site]
    print(f"{site:<15}{counts['VX.1']:>6}{counts['VX.2']:>7}{counts['VX.2.1']:>7}")
print(f"dominant_lineage {dominant_lineage}")
print("valid_lineage_counts " + str({lineage: valid_lineage_counts[lineage] for lineage in lineage_order}))

collection_site   VX.1   VX.2 VX.2.1
HarborLab           0      1      3
NorthClinic         2      1      1
RidgeMobile         0      0      1
dominant_lineage VX.2.1
valid_lineage_counts {'VX.1': 2, 'VX.2': 2, 'VX.2.1': 5}


In [7]:
print("bar_chart_placeholder lineage_counts_by_site")
for site in site_order:
    segments = []
    for lineage in lineage_order:
        count = lineage_counts_by_site[site][lineage]
        segments.append(f"{lineage}:{'#' * count if count else '-'}")
    print(f"{site:<12} " + " ".join(segments))

bar_chart_placeholder lineage_counts_by_site
HarborLab    VX.1:- VX.2:# VX.2.1:###
NorthClinic  VX.1:## VX.2:# VX.2.1:#
RidgeMobile  VX.1:- VX.2:- VX.2.1:#


## Interpretation

After excluding the three low-coverage RidgeMobile specimens, `VX.2.1` is the dominant lineage in the synthetic surveillance set. HarborLab contributes most of the `VX.2.1` detections, while NorthClinic retains a mixed profile with `VX.1`, `VX.2`, and `VX.2.1`. I would prioritize resequencing `VLA-S004`, `VLA-S009`, and `VLA-S012` before using RidgeMobile prevalence as an outbreak decision point.